# Analyzing Punctuation Patterns

There are many ways to expose punctuation patterns using Lexos, but no Lexos module has a workflow focused on analyzing their distribution in a corpus. This tutorial provides a workflow using a dedicated Python class intended to work alongside the existing Lexos toolkit.

In [ ]:
# Import the StructuralAnalyzer class
from lexos.structural_stylometry import StructuralAnalyzer as sa

## Get the Corpus

The `StructuralAnalyzer` class takes a corpus of texts in the form of a dictionary of labels (doc_ids) and raw strings or spaCy `Docs`. If a documents are strings, they are converted internally to spaCy `Doc` objects using the default "xx_sent_ud_sm" model unless you specify the model when you instantiate the `StructuralAnalyzer` object.

In the example below, we will set up a sample corpus consisting of four texts. Notice that the sample documents below contain a variety of whitespace combinations. These can optionally be recorded as tokens as a pre-processing step before the texts are converted to `Doc` objects. This can be useful if you are submitting texts with irregularities or unusual formatting such as might be produced by OCR.

In [ ]:
# Raw text string sample containing multi-author mock texts
sample_corpus = {
    "AuthorA_Doc1": "Thus, it came to pass; the empire fell,  not by sword, but by time. \n",
    "AuthorA_Doc2": "Therefore, the king wept;  no crown could save him, nor gold. \n",
    "AuthorB_Doc1": "The empire fell! \n\n Time conquered all! Swords were useless!",
    "AuthorB_Doc2": "Victory was lost! \n\n All hope vanished! Shields shattered!",
}

### Using a Lexos `Loader`

When importing data from files, it can be useful to load them with the Lexos `Loader` class. An example is given below; however, instead of calling the `load()` method, we will simply add the data from the sample corpus we created above to the `Loader` object. 

In [ ]:
# Import the Loader class and create an instance
from lexos.io.loader import Loader
loader = Loader()

# Uncomment this to use real files
# loader.load("path/to/directory_of_files")

# Add our sample_corpus from above
for name, doc in sample_corpus.items():
    loader.names.append(name)
    loader.texts.append(doc)

# You would now instantiate the analyzer with the loader object
# analyzer = sa(corpus=loader, ...)

### Using a Lexos `Corpus`

If you have a Lexos `Corpus`, you can also pass it to the `Analyzer`. Here is an example in which we create a `Corpus` and add the data from the sample corpus we created above. 

In [ ]:
# Import the Python tempfile module and the Lexos Corpus class
import tempfile
from lexos.corpus import Corpus

# Create a temporary directory for the corpus
corpus_directory = tempfile.mkdtemp()  # Or use a permanent directory like "./my_corpus"

# Create the corpus
lexos_corpus = Corpus(
    name="My Research Corpus",
    corpus_dir=corpus_directory
)

# Add the data
for name, content in sample_corpus.items():
    lexos_corpus.add(
        name=name,
        content=content,
        is_active=True,
    )

# You would now instantiate the analyzer with the Corpus object
# analyzer = sa(corpus=lexos_corpus, ...)

## Create the Analyzer Object

The `StructuralAnalyzer` object is instantiated with your data and a variety of settings:

`min_punctuation_threshold`: The minumum number of punctuation marks required per document. The default is 20.
`action_on_low_count`: The action taken if a document falls below the `min_punctuation_threshold`. The default "warn" sends a warning to the console. Set to "drop" if you want the document to be dropped from the analysis.
`max_features`: The maximum number of features to track in the vocabulary. The default is 100.
`include_whitespace`: Whether to include whitespace markers as features. The default is `True`.
`feature_mode`: The default "all" tracks all tokens in the corpus. Set to "structural_only" to include only punctuation and spacing or to "punctuation_only" to include only punctuation.

In [ ]:
analyzer = sa(
    corpus=lexos_corpus,
    model="xx_sent_ud_sm",
    min_punctuation_threshold=1,
    action_on_low_count="warn",
    max_features=15,
    include_whitespace=True,
    feature_mode="structural_only" # This mode includes punctuation and spacing, but excludes words
)

### View the Vocabulary

If you set `include_whitespace` to `True`, your documents are pre-processed to convert double spaces, multiple newlines, and trailing whitespace to `[WS_DOUBLE_SPACE]`, `[WS_MULTIPLE_NEWLINE]`, and `[WS_TRAILING_SPACE]`. This allows them to be counted as tokens when the documents are converted to spaCy `Doc` objects.

In many cases, you will not want these features to be counted because they may be indicators of OCR errors or other formatting irregularities, rather than stylistically significant features. It is thus a good idea to inspect the vocabulary in the the `StructuralAnalyzer` object before proceeding.

In [ ]:
analyzer.vocabulary

## Build the Matrix

We're now build a frequency matrix to represent our corpus data. By default, the `get_feature_matrix` method uses Term Frequency-Inverse Document Frequency (TF-IDF), which we specify explicitly in the code cell below.

In [ ]:
matrix = analyzer.get_feature_matrix(method="tfidf")
matrix

We can also supply `method="raw"` to get raw counts or `method="burrows-z"` to get a matrix of Z-scores.

### Choosing a Representation Method

Using raw counts can be statistically misleading if your documents vary in length. For this reason, the default method is "tfidf". However, because punctuation occurs at vastly higher frequencies than individual words, standard TF-IDF can occasionally over-penalize punctuation marks across a small corpus. To fix this, we use raw instead of log-scaled term frequencies (TF), combined with standard Inverse Document Frequency (IDF).

For an alternative approach, we can use the Burrows' Z method. This method normalizes the raw relative frequencies into standard deviations away from the corpus mean (Z-scores). The output is a matrix of shape (`number_of_docs`, `number_of_features`).

You can also view the matrix as a pandas DataFrame:

In [ ]:
analyzer.to_df(method="tfidf")

## Get Delta Distances

A common way to use such frequency data is to transform the matrix into a distance matrix to analyse vector similarities in documents.

There are many metrics to calculate distances between features of a frequency matrix. We'll start with the Classic Burrows' Delta, which calculates the Manhattan distance (mean absolute difference) between the Z-scores of two documents. The output is a pairwise distance matrix of shape (`number_of_docs`, `number_of_docs`).

You can view the distance between two documents by indexing into the matrix. For example, `delta_distances[0, 1]` gives the distance between the first and second documents in the corpus.

In [ ]:
delta_distances = analyzer.get_distance_matrix(method="classic")

print("\n--- Burrows' Delta Distance Matrix ---")
print(f"Distance between Text 1 & Text 2 (Same Author): {delta_distances[0, 1]:.4f}")
print(f"Distance between Text 1 & Text 3 (Diff Author): {delta_distances[0, 2]:.4f}")

A smaller Delta value implies closer stylistic alignment. In the example output, `delta_distances[0, 1]` yields a much lower numerical distance than `delta_distances[0, 2]`. The engine relies heavily on punctuation layout choices to recognize that Author A wrote the first two documents, separating them from the exclamation-heavy habits of Author B.

You can also implement the Argamon quadratic variant, which handles the high standard deviations introduced by mixed token types much better than the original Z-score cityblock (Manhattan) distance.

In [ ]:
quad_delta_distances = analyzer.get_distance_matrix(method="quadratic")

print("\n--- Argamon’s Quadratic Delta Distance Matrix ---")
print(f"Distance between Text 1 & Text 2 (Same Author): {quad_delta_distances[0, 1]:.4f}")
print(f"Distance between Text 1 & Text 3 (Diff Author): {quad_delta_distances[0, 2]:.4f}")

The output of `get_distance_matrix` can also be viewed as a pandas DataFrame:

In [ ]:
quad_delta_distances = analyzer.get_distance_matrix(method="quadratic", as_df=True)
quad_delta_distances

## Plot and Analyze

Distance matrices can be difficult for humans to process, so it is common to plot visualizations which capture their qualities. This can allow us to view exactly which punctuation marks, words, or whitespace behaviors are discriminating features in our documents.

The `visualize` method generates a dendrogram and a Principal Component Analysis (PCA) plot showing document similarity. The layout of the PCA plot is driven by PCA components (loadings) that represent the linear combination of the original features. A high positive or negative loading means that feature heavily influences where a text lands along that specific PCA axis.

We build these plots with the `visualize` function. The function takes the method and the top number of features to show for each principal component. Optionally, you can hide the plots or the loadings with `show_plots` and `show_loadings` (by default, set to `True`).

By default, the function will use Euclidean calculations with "ward" linkage. If you specify `method="burrows_z"`, it will use Manhattan distance with average linkage.

In [ ]:
analyzer.visualize(method="burrows_z", top_n=3, show_plots=True, show_loadings=True)

Should you need to access the loadings directly, you can call `get_loadings`, which returns a dictionary with "PC1" and "PC2" as the keys, and pandas DataFrames as the values.

In [ ]:
loadings = analyzer.get_loadings(method="burrows_z")
display(loadings["PC1"])

## Export the Results

The `to_csv` function allows you to export your matrix to a CSV file in a format compatible with R's `stylo` package: The generated layout places variables as columns and observations as rows. If you are feeding this into standard R workflows, this aligns directly with standard `read.csv(file, row.names=1)` matrix construction templates.

In [ ]:
filepath = "burrows_z_loadings.csv"

# Uncomment to save the file
# analyzer.to_csv(filepath=filepath, method="burrows_z")